# Day 22 · 因果推断：聊天机器人使用能减轻老年人的孤独吗

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 用潜在结果框架说明为什么观察数据的比较会有偏差
- 在模拟数据里用回归调整和逆概率加权恢复真实效应
- 说清这些方法的前提：无未测混杂、重叠性
- 用双重差分分析服务层面的上线效果，并判断它能否支持因果结论

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 观察数据：6000 位老年人

In [2]:
from course.causal import simulate_chatbot_use, naive_difference, regression_adjusted, ipw_ate, did_estimate
obs = simulate_chatbot_use(6000, true_effect=-2.0, seed=1)
obs.groupby("use")[["age", "literacy", "lonely0", "lonely1"]].mean().round(2)

,age,literacy,lonely0,lonely1
use,,,,
0,70.42,-0.32,38.29,38.75
1,69.24,0.46,42.43,39.70


**练习 22.1**：使用者和未使用者的基线孤独感（`lonely0`）差多少？（使用者减未使用者）这说明了什么？

In [3]:
_g = obs.groupby("use").lonely0.mean()
ans_22_1 = None
check("练习 22.1", ans_22_1, float(_g[1] - _g[0]), tol=1e-6)

[ 待完成 ] 练习 22.1


## 2. 各种估计

In [4]:
est = pd.Series({
    "naive": naive_difference(obs),
    "adjust age + literacy": regression_adjusted(obs, ["age", "literacy"]),
    "adjust + baseline loneliness": regression_adjusted(obs, ["age", "literacy", "lonely0"]),
    "IPW": ipw_ate(obs, ["age", "literacy", "lonely0"]),
}); est.round(3)

naive                           0.950
adjust age + literacy           1.863
adjust + baseline loneliness   -1.948
IPW                            -1.665
dtype: float64

**练习 22.2**：哪个估计离真实效应（−2）最远？填它在上表中的名字。

In [5]:
ans_22_2 = None
check("练习 22.2", ans_22_2, (est + 2).abs().idxmax())

[ 待完成 ] 练习 22.2


## 3. 随机化的版本

In [6]:
rct = simulate_chatbot_use(6000, true_effect=-2.0, randomised=True, seed=2)
print("RCT 组间差:", round(naive_difference(rct), 3))

RCT 组间差: -1.977


**练习 22.3**：在 RCT 数据上，把基线孤独感作为协变量调整（`regression_adjusted(rct, ["lonely0"])`）。估计值仍应接近 −2，但标准误更小。用 bootstrap(200 次）比较未调整与调整后估计的标准差，填入**调整后**的标准差。

In [7]:
rng = np.random.default_rng(0)
b_raw, b_adj = [], []
for _ in range(200):
    s = rct.sample(len(rct), replace=True, random_state=int(rng.integers(1e9)))
    b_raw.append(naive_difference(s)); b_adj.append(regression_adjusted(s, ["lonely0"]))
print("未调整 SD:", round(np.std(b_raw), 3), " 调整后 SD:", round(np.std(b_adj), 3))
ans_22_3 = None
check("练习 22.3", ans_22_3, float(np.std(b_adj)), tol=1e-6)

未调整 SD: 0.194  调整后 SD: 0.112
[ 待完成 ] 练习 22.3


## 4. 双重差分：社区中心上线聊天机器人

10 家社区中心上线了心理支持聊天机器人，10 家没有。结局是每月主动求助的人数（每家平均）。

In [8]:
pre_t, post_t = 42.0, 55.0     # 上线中心:上线前、后
pre_c, post_c = 40.0, 46.0     # 对照中心

**练习 22.4**：计算双重差分估计。然后写一句话：要相信这个数字，需要什么前提？

In [9]:
ans_22_4 = None
check("练习 22.4", ans_22_4, did_estimate(pre_t, post_t, pre_c, post_c), tol=1e-9)

[ 待完成 ] 练习 22.4


*前提：*

## 口述题（录音，每题 90 秒）

1. 如何判断评测分数提升是因为模型变好，而不是评测集被污染？
   <details><summary>参考要点</summary>对照实验：同时测原题和改写题、旧题和训练截止后的新题；如果提升只出现在原题或旧题上，怀疑污染。还要检查同时改动的其他设置，例如裁判版本、提示词。</details>
2. 线上 A/B 测试之外，怎么评估聊天机器人对用户的影响？
   <details><summary>参考要点</summary>准实验方法：双重差分、断点回归、倾向得分方法，并说明各自的前提；观察数据结论要谨慎。</details>

## 研究延伸

你的 UTAUT2 研究关心谁会采纳心理健康应用。把它和效果研究结合起来，可以回答一个重要问题：观察数据中的“使用者效果”有多少来自自我选择？这是数字心理健康研究中常见但常被忽视的偏差来源。

In [10]:
summary()

本 notebook 自动批改:0 / 4 通过
